# NeuroQueue training
Runs the same reproducible pipeline as `python -m ml.train` on a free Kaggle / Colab GPU.

1. Clone or upload this repository.
2. Put the dataset under `ml/data/raw/` in a folder-per-class layout (`glioma`, `meningioma`, `pituitary`, `notumor`). On Kaggle, add the *Brain Tumor MRI Dataset* and point `--data` at `/kaggle/input/brain-tumor-mri-dataset`.
3. Run all cells. Artifacts are written to `ml/artifacts/`.

In [ ]:
!pip -q install onnx matplotlib
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
# De-duplicate (perceptual hash) and split. Reports how many duplicates were removed.
from pathlib import Path
from ml.data import build_splits
DATA = Path('ml/data/raw')   # Kaggle: Path('/kaggle/input/brain-tumor-mri-dataset')
print(build_splits(DATA, Path('ml/artifacts'))['report'])

In [ ]:
# Train, calibrate on validation, tune thresholds on validation, evaluate once on test, export ONNX, pick demo scans.
!python -m ml.train --data ml/data/raw --epochs 12

In [ ]:
import json
m = json.load(open('ml/artifacts/metrics.json'))
print('test accuracy', m['test']['accuracy'])
print(json.dumps(m['test']['per_class'], indent=2))
print(json.dumps(m['test']['verifier'], indent=2))
print(open('ml/artifacts/thresholds.json').read())

In [ ]:
from IPython.display import Image, display
display(Image('ml/artifacts/reliability.png')); display(Image('ml/artifacts/confusion_matrix.png'))